# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [3]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on github we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.
(We will share the file with you on Slack!)


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv('DATABASE')
USER_DB = os.getenv('USER_DB')
PASSWORD = os.getenv('PASSWORD')
HOST = os.getenv('HOST')
PORT = os.getenv('PORT')


The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [15]:
print(DATABASE)
print(USER_DB)
print(PASSWORD)
print(HOST)
print(PORT)

# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE,
    user=USER_DB,
    password=PASSWORD,
    host=HOST,
    port=PORT
)

postgres
aiengonleng090226
mialovesicecream
ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com
5432


### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commmands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [16]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [17]:
cur.execute('SELECT * FROM eda.king_county_house_sales LIMIT 10')
cur.fetchall()

[(datetime.date(2014, 10, 13), 221900.0, 7129300520, 1),
 (datetime.date(2014, 12, 9), 538000.0, 6414100192, 2),
 (datetime.date(2015, 2, 25), 180000.0, 5631500400, 3),
 (datetime.date(2014, 12, 9), 604000.0, 2487200875, 4),
 (datetime.date(2015, 2, 18), 510000.0, 1954400510, 5),
 (datetime.date(2014, 5, 12), 1230000.0, 7237550310, 6),
 (datetime.date(2014, 6, 27), 257500.0, 1321400060, 7),
 (datetime.date(2015, 1, 15), 291850.0, 2008000270, 8),
 (datetime.date(2015, 4, 15), 229500.0, 2414600126, 9),
 (datetime.date(2015, 3, 12), 323000.0, 3793500160, 10)]

With `conn.close()` you can close the connection again.

In [18]:
#close the connection
conn.close()

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [19]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE,
    user=USER_DB,
    password=PASSWORD,
    host=HOST,
    port=PORT
)

In [20]:
# import the data into a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales LIMIT 10"
df_psycopg = pd.read_sql(query_string, conn)

C:\Users\blabl\AppData\Local\Temp\ipykernel_19668\1176423828.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_psycopg = pd.read_sql(query_string, conn)


In [37]:
#close the connection
conn.close()

In [22]:
df_psycopg.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


In [ ]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE,
    user=USER_DB,
    password=PASSWORD,
    host=HOST,
    port=PORT
)

# import the king_county_house_sales data into a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales"
df_psycopg = pd.read_sql(query_string, conn)

#export the data to a csv-file
df_psycopg.to_csv('data/king_county_house_sales.csv',index=False, sep=';')

# import the  king_county_house_sales data data into a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_details"
df_psycopg = pd.read_sql(query_string, conn)

#export the data to a csv-file
df_psycopg.to_csv('data/king_county_house_details.csv',index=False, sep=';')

C:\Users\blabl\AppData\Local\Temp\ipykernel_19668\1850511117.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_psycopg = pd.read_sql(query_string, conn)
C:\Users\blabl\AppData\Local\Temp\ipykernel_19668\1850511117.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_psycopg = pd.read_sql(query_string, conn)


In [38]:
conn.close()

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database sting (a link that includes every information we entered in the conn object)

In [39]:
from sqlalchemy import create_engine

#read the database string from the .env
load_dotenv()

DB_STRING = os.getenv('DB_STRING')

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [40]:
#import the data to a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales"
df_sqlalchemy = pd.read_sql(query_string, db)
df_sqlalchemy.to_csv('king_county_house_sales.csv',index=False, sep=';')

query_string = "SELECT * FROM eda.king_county_house_details"
df_sqlalchemy = pd.read_sql(query_string, db)
df_sqlalchemy.to_csv('king_county_house_details.csv',index=False, sep=';')

In [13]:
df_sqlalchemy.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [ ]:
#export the data to a csv-file
df_sqlalchemy.to_csv('eda.csv',index=False)

In [30]:
#import the data from a csv-file
df_import = pd.read_csv('data/king_county_house_details.csv')

df_sales = pd.read_csv('data/king_county_house_sales.csv')

In [28]:
df_import.view.unique()

array([ 0.,  3.,  2.,  4.,  1., nan])

In [27]:
df_import

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,1000102,6.0,3.00,2400.0,9373.0,2.0,NaN,0.0,3,7,2400.0,0.0,1991,0.0,98002,47.3262,-122.214,2060.0,7316.0
1,100100050,3.0,1.00,1320.0,11090.0,1.0,0.0,0.0,3,7,1320.0,0.0,1955,0.0,98155,47.7748,-122.304,1320.0,8319.0
2,1001200035,3.0,1.00,1350.0,7973.0,1.5,NaN,0.0,3,7,1350.0,0.0,1954,0.0,98188,47.4323,-122.292,1310.0,7491.0
3,1001200050,4.0,1.50,1260.0,7248.0,1.5,NaN,0.0,5,7,1260.0,0.0,1955,NaN,98188,47.4330,-122.292,1300.0,7732.0
4,1003000175,3.0,1.00,980.0,7606.0,1.0,0.0,0.0,3,7,980.0,0.0,1954,0.0,98188,47.4356,-122.290,980.0,8125.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21415,993002177,3.0,2.50,1380.0,1547.0,3.0,0.0,0.0,3,8,1380.0,0.0,2000,NaN,98103,47.6908,-122.341,1380.0,1465.0
21416,993002225,3.0,2.25,1520.0,1245.0,3.0,NaN,0.0,3,8,1520.0,0.0,2004,0.0,98103,47.6907,-122.340,1520.0,1470.0
21417,993002247,3.0,2.25,1550.0,1469.0,3.0,0.0,0.0,3,8,1550.0,0.0,2004,0.0,98103,47.6911,-122.341,1520.0,1465.0
21418,993002325,2.0,1.50,950.0,4625.0,1.0,0.0,0.0,4,7,950.0,0.0,1949,NaN,98103,47.6912,-122.340,1440.0,4625.0


In [32]:
df_sales.dtypes

date         object
price       float64
house_id      int64
id            int64
dtype: object

In [35]:
df_import[df_import['yr_renovated'] > 2014]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
13,1005000036,3.0,1.75,1840.0,8601.0,1.0,0.0,0.0,3,7,920.0,920.0,1905,20140.0,98118,47.5359,-122.276,1390.0,7452.0
58,1023089140,3.0,2.00,1740.0,41275.0,1.0,0.0,0.0,3,8,1740.0,0.0,1974,19890.0,98045,47.4914,-121.763,2630.0,41275.0
65,1024039049,3.0,2.50,2920.0,34527.0,1.0,0.0,4.0,4,9,1800.0,1120.0,1954,19830.0,98116,47.5799,-122.400,2480.0,7933.0
77,1025039145,4.0,2.00,3140.0,10875.0,1.0,0.0,1.0,3,7,1940.0,1200.0,1939,19690.0,98199,47.6656,-122.406,3300.0,10080.0
183,1062100100,4.0,2.00,2100.0,4857.0,2.0,0.0,0.0,3,8,2100.0,0.0,1965,19840.0,98155,47.7521,-122.279,1450.0,5965.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21239,9828200746,2.0,1.50,1120.0,1024.0,2.0,0.0,0.0,3,8,1120.0,0.0,1970,19980.0,98122,47.6175,-122.298,1120.0,1549.0
21260,9828700200,4.0,3.00,2170.0,4000.0,2.0,0.0,0.0,4,9,1610.0,560.0,1982,20110.0,98112,47.6196,-122.292,1670.0,4000.0
21273,9828701690,3.0,2.00,1530.0,3400.0,1.0,0.0,0.0,3,7,990.0,540.0,1907,20140.0,98112,47.6204,-122.296,1880.0,4212.0
21306,9829200250,3.0,2.00,2600.0,6600.0,2.0,0.0,4.0,3,10,1930.0,670.0,1970,20140.0,98122,47.6055,-122.285,2670.0,6270.0


In [36]:
df_import['yr_renovated'] > 2014

0        False
1        False
2        False
3        False
4        False
         ...  
21415    False
21416    False
21417    False
21418    False
21419    False
Name: yr_renovated, Length: 21420, dtype: bool